# 6 自定义量化 A8W8 matmul 算子开发并接入 Qwen3-8B

本章是整个启航营的**核心实践环节**，将学习到的所有知识沉淀转化为一次完整的自定义算子开发实战。

实践的环境准备、提交方式可参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

### 本章定位

本章将开发一个自定义的量化 matmul 算子 `QmmCustom`，通过 A8W8 量化（Activation INT8 + Weight INT8）减少计算量和内存占用，从而提升推理性能。这是本次启航营中综合性最强的一章——涵盖了从算子概念理解、Tiling 设计、Kernel 实现到算子编译、接入网络与功能性能测试的全链路。

### 通过实践达成的学习目标

- **理解量化推理原理**：掌握 A8W8 量化方案（INT8 激活 + INT8 权重）的数学原理与 Scale 反量化机制，理解 perChannelScale 与 perTokenScale 的作用。
- **设计 Tiling 数据结构与函数**：根据算子原型定义自行设计 TilingData 结构体，实现多核分块的 Tiling 函数，理解 Cube 计算单元的分块策略。
- **实现 Cube+Vector 双路径 Kernel**：完成 Cube-only（INT32 输出）和 Cube+Vector 协作（反量化 BF16 输出）两条 Kernel 路径的实现，理解昇腾 AI Core 的 Cube 与 Vector 协作编程模型。
- **完成算子编译与 Torch 接入**：编译自定义算子并通过 Torch 接口接入，打通“算子开发 → 框架调用”的链路。
- **测试算子功能与性能，尝试优化迭代**：验证自定义算子的精度正确性，测试单算子与网络性能，并根据测试结果对 Tiling 或 Kernel 进行优化迭代。

### 本章大纲

- 环境准备
- A8W8 量化 matmul 算子概念
- 自定义算子实现
    - **任务一：自行设计tilingdata结构体并实现tiling函数**
    - **任务二：实现A8W8输入，INT32输出的kernel**
    - **任务三：实现A8W8输入，经过perChannelScale和perTokenScale反量化输出BF16的kernel**
- 编译自定义算子
    - **任务四：成功编译自定义算子**
- 测试自定义算子功能性能
    - **任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程**
- 自定义算子接入模型
- 模型性能测试
    - **任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程**
- 提交要求
---

## 1 环境准备

定位教程目录、准备运行目录，并把 CANN 环境变量导入当前 Notebook 进程：

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

print('[环境准备] 开始', flush=True)

def locate_repo_root():
    repo_roots = []
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'tutorials/llm_inference/qwen3_8b/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'tutorials' / 'llm_inference' / 'qwen3_8b'
TUTORIAL_SRC_DIR = TUTORIAL_DIR / 'src'
if str(TUTORIAL_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_SRC_DIR))
SRC_DIR = TUTORIAL_DIR / 'src' / 'op_custom'

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment.')
print('CANN包路径 =', os.environ.get('ASCEND_TOOLKIT_HOME'))
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' in line:
        os.environ.__setitem__(*line.split('=', 1))

RECIPE_ROOT = TUTORIAL_DIR / 'src/inference_scripts/recipe_qwen3_8b'
QMM_CUSTOM_DIR = SRC_DIR / 'qmm_custom'

print('教程目录 =', TUTORIAL_DIR)
print('推理模型目录 =', RECIPE_ROOT)
print('算子相关文件目录 =', SRC_DIR)
print('算子实现目录 =', QMM_CUSTOM_DIR)
print('[环境准备] 完成', flush=True)

[环境准备] 开始
CANN包路径 = /home/developer/Ascend/cann-8.5.2
教程目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b
推理模型目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b
算子相关文件目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom
算子实现目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom
[环境准备] 完成


安装所需的依赖：

In [2]:
%pip install -r {SRC_DIR}/requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple --trusted-host pypi.tuna.tsinghua.edu.cn

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


---
## 2 A8W8 量化 matmul 算子概念

### 2.1 什么是 A8W8 量化

A8W8 量化是 LLM 推理中常用的一种量化方案：
- **A（Activation）**：将输入激活值从 BF16/FP16 量化为 **INT8**
- **W（Weight）**：将权重从 BF16/FP16 量化为 **INT8**
- **反量化**：计算结果 `INT8 × INT8 → INT32` 后，通过 Scale 反量化回浮点精度

### 2.2 算子规格

在上一章中，通过分析量化模型的profiling，归纳出了自定义算子的输入输出规格：

| 算子名称 | QmmCustom |
|------|-----|
| 输入 x1 (A) | INT8, ND 格式, shape [M, K] |
| 输入 x2 (B) | INT8, **FRACTAL_NZ** 格式, shape [K, N] |
| 输入 scale | FLOAT32, ND 格式, shape [N] |
| 输入 pertoken_scale | FLOAT32, ND 格式, shape [M]（可选） |
| 输出 y | 有 perTokenScale → BF16，无 → INT32；ND 格式, shape [M, N] |
| 架构 | ascend910b |

**注意**：x2 权重矩阵使用 **FRACTAL_NZ** 格式，这是昇腾 AI Core Cube 单元的高效存储格式，能提升 matmul 计算性能。

### 2.3 Scale 模式

QmmCustom 算子支持两种 Scale：

| Scale 类型 | Shape | 作用 | 
|-----------|-------|------|
| **perChannelScale** | `[N]` | 对每个输出通道单独缩放 |
| **perTokenScale** | `[M]` | 对每行输入单独缩放 |

计算公式：
```
result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale
```

## 3 自定义算子实现

与前面学习过的内容一致，本次实践采用 `<<<>>>` Kernel 直调方式开发上述原型的量化 matmul 算子。直调方式通过 `kernel<<<numBlocks, dynUBufSize, stream>>>(args)` 语法在 Host 侧直接启动 Kernel，适合快速验证算子功能与性能。

本节包含如下三个任务：

### 任务一：自行设计 TilingData 结构体并实现 Tiling 函数

根据算子原型与 Kernel 的实现需求，自行设计 `QmmCustomTilingData` 结构体，并实现 `CalcQmmTiling` 函数。Tiling 函数需根据输入的 M、N、K 维度，结合平台硬件资源（L1/L0A/L0B/UB 内存限制、可用核数），计算多核分块参数。可借助 `platform_ascendc::PlatformAscendCManager` 获取硬件信息。

> **Tiling 设计提示**：
> 
> 设计 TilingData 时需要考虑以下要素：
> 1. **`TCubeTiling`**：CANN Matmul 库提供的标准 Tiling 结构体，包含 M/N/K 的分块大小（`baseM`、`baseN`、`baseK`）、使用的核数（`usedCoreNum`）等字段。可通过 `GetMatmulTiling` API 自动计算。
> 2. **`isPertoken`**：标记是否走反量化路径，用于 Kernel 内部分支选择。
> 3. **`workspaceSize`**：Path 2 中 Cube 写入 INT32 中间结果所需的 GM workspace 大小。
> 4. **多核分块策略**：将 M 维度或 N 维度分配到多个核上并行计算，每个核处理一个分块。分块大小需平衡 L1/L0A/L0B 缓存利用率与多核负载均衡。

### 任务二：实现 A8W8 输入、INT32 输出的 Kernel

实现 `QmmCubeBasicKernel` 类，完成纯 Cube 路径的 INT8 矩阵乘法计算。

计算公式： ```result_int32 = x1_int8 @ x2_int8```

### 任务三：实现 A8W8 输入、经 perChannelScale 和 perTokenScale 反量化输出 BF16 的 Kernel

实现 `QmmPertokenKernel` 类，在 Cube 计算的基础上增加 Vector 反量化路径。

计算公式： ```result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale```

其中 `perChannelScale` 沿 N 维广播，`perTokenScale` 沿 M 维广播，两者均为 FLOAT32 类型。

---

### 执行路径说明

在 Host 侧的 `qmm_custom` 函数中，根据是否传入 `pertoken_scale` 设置 `isPertoken` 标志，Kernel 启动时据此选择执行路径：

| 路径 | 条件 | 计算流程 | 输出类型 |
|------|------|----------|----------|
| **Path 1: Cube-only** | `isPertoken == 0` | INT8 matmul → INT32 | INT32 |
| **Path 2: Cube+Vector** | `isPertoken == 1` | INT8 matmul → INT32 → dequant(scale×pertoken) → BF16 | BF16 |

**Path 1（Cube-only）**：仅使用 AI Core 的 Cube 计算单元，完成 INT8×INT8 矩阵乘法后直接将 INT32 结果写入输出 GM，无 Vector 参与。对应 `QmmCubeBasicKernel` 的实现。

**Path 2（Cube+Vector）**：Cube 单元先完成 INT8 matmul 得到 INT32 中间结果并写入 GM workspace；随后 Vector 单元从 GM 读取 INT32 结果，依次完成 `Cast`（INT32→FLOAT32）、乘以 `perChannelScale` 和 `perTokenScale`、`Cast`（FLOAT32→BF16）等反量化操作，最终将 BF16 结果写入输出 GM。对应 `QmmPertokenKernel` 的实现。

Kernel 入口函数 `qmm_custom_kernel` 使用 `__mix__(1, 2)` 属性声明，表示为每个 Block 分配 1 个 Cube 核和 2 个 Vector 核，使 Cube 与 Vector 可在同一 Kernel 中协同工作。Kernel 内部通过 `tilingData.isPertoken` 判断走哪条路径，两条路径复用相同的 Tiling 结构和 Kernel 启动入口。

Tiling、Kernel、Torch 接口的实现均位于 `qmm_custom.asc` 文件中。文件已给出整体框架与函数签名，请自行完成下方代码中的 TODO 部分，以实现完整的自定义算子：

In [3]:
%%writefile {QMM_CUSTOM_DIR}/qmm_custom.asc
 #include "acl/acl_base.h"
 	 #include "kernel_operator.h"
 	 
 	 // CANN 8.5.x 的自定义 C/V 混合模式要求在包含 Matmul 接口前定义该宏。
 	 #define ASCENDC_CUBE_ONLY
 	 #include "lib/matmul_intf.h"
 	 
 	 #include "tiling/platform/platform_ascendc.h"
 	 #include "tiling/tiling_api.h"
 	 #include "torch_npu/csrc/core/npu/NPUStream.h"
 	 #include <torch/extension.h>
 	 
 	 #include <algorithm>
 	 #include <cmath>
 	 #include <cstdint>
 	 #include <limits>
 	 
 	 using namespace AscendC;
 	 
 	 namespace {
 	 
 	 // Cube 矩阵计算单元M维度最小对齐粒度，固定16个元素
 	 constexpr uint32_t kCubeMAlign = 16;
 	 // INT8 FRACTAL_NZ 权重矩阵B内轴N维度硬件对齐单位32
 	 // 所有N分块、分块起始偏移必须按32对齐，否则硬件访存异常
 	 constexpr uint32_t kNzNAlign = 32;
 	 // K维度FRACTAL_NZ格式对齐粒度32
 	 constexpr uint32_t kNzKAlign = 32;
 	 // 临时工作空间内存对齐字节数512
 	 constexpr uint32_t kWorkspaceAlign = 512;
 	 // AIV向量单元单次向量化计算最大元素数量
 	 constexpr uint32_t kVecTileElems = 1024;
 	 // AIC Cube计算完成后通知AIV向量核的跨核同步标记
 	 constexpr uint16_t kCubeToVectorFlag = 0x8;
 	 
 	 // 向上取整除法，无符号32位整数专用
 	 inline uint32_t CeilDivU32(uint32_t x, uint32_t y) {
 	   return (x + y - 1U) / y;
 	 }
 	 
 	 // 将数值向上对齐到align整数倍，32位无符号
 	 inline uint32_t AlignUpU32(uint32_t x, uint32_t align) {
 	   return CeilDivU32(x, align) * align;
 	 }
 	 
 	 // 将数值向上对齐到align整数倍，64位无符号（内存偏移专用）
 	 inline uint64_t AlignUpU64(uint64_t x, uint64_t align) {
 	   return ((x + align - 1U) / align) * align;
 	 }
 	 
 	 // 多核网格切分规划结果结构体，存储M/N分块策略与核占用信息
 	 struct GridPlan {
 	   uint32_t singleM;      // 单个逻辑核处理的M维度块大小（对齐后）
 	   uint32_t singleN;      // 单个逻辑核处理的N维度块大小（对齐后）
 	   uint32_t mBlocks;      // M维度总分块数量
 	   uint32_t nBlocks;      // N维度总分块数量
 	   uint32_t usedCores;    // 当前方案占用AIC逻辑核总数
 	 };
 	 
 	 // 自动选择最优多核切分网格策略
 	 // 规则说明：
 	 // 1. decode小M场景仅切N，避免单核M过小导致Cube利用率低
 	 // 2. prefill大M场景二维切M+N，最大化AIC硬件并行度
 	 // 3. M块强制16对齐、N块强制32对齐，适配INT8 FRACTAL_NZ硬件格式约束
 	 static GridPlan ChooseGrid(uint32_t M, uint32_t N, uint32_t maxCores) {
 	   // 初始化最优方案：整块不切分
 	   GridPlan best{AlignUpU32(M, kCubeMAlign), AlignUpU32(N, kNzNAlign),
 	                 1U, 1U, 1U};
 	   uint32_t bestUsed = 0;
 	   double bestPenalty = std::numeric_limits<double>::infinity();
 	 
 	   // M<=64时不切M，仅沿N维度并行；大M场景允许M多切分
 	   const uint32_t maxMParts = (M <= 64U) ? 1U : maxCores;
 	   // 遍历所有可行M切分份数
 	   for (uint32_t requestedMParts = 1; requestedMParts <= maxMParts;
 	        ++requestedMParts) {
 	     // 计算每份M块大小并向上对齐16
 	     const uint32_t singleM =
 	         AlignUpU32(CeilDivU32(M, requestedMParts), kCubeMAlign);
 	     // 实际M分块总数（含尾块）
 	     const uint32_t actualMParts = CeilDivU32(M, singleM);
 	 
 	     // 遍历所有可行N切分份数
 	     for (uint32_t requestedNParts = 1; requestedNParts <= maxCores;
 	          ++requestedNParts) {
 	       // 计算每份N块大小并向上对齐32
 	       const uint32_t singleN =
 	           AlignUpU32(CeilDivU32(N, requestedNParts), kNzNAlign);
 	       // 实际N分块总数（含尾块）
 	       const uint32_t actualNParts = CeilDivU32(N, singleN);
 	       // 当前方案占用总逻辑核数
 	       const uint32_t used = actualMParts * actualNParts;
 	       // 核数非法：跳过
 	       if (used == 0U || used > maxCores) {
 	         continue;
 	       }
 	 
 	       // 硬件最优Cube块接近128x256，期望 singleN ≈ 2*singleM，偏离越大惩罚越高
 	       const double aspect =
 	           static_cast<double>(singleN) /
 	           (2.0 * static_cast<double>(std::max(singleM, 1U)));
 	       const double aspectPenalty = std::fabs(std::log2(std::max(aspect, 1e-9)));
 	       // 同核数下，单核输出面积越小越好，降低尾块、中间缓存延迟
 	       const double areaPenalty =
 	           static_cast<double>(singleM) * static_cast<double>(singleN) * 1e-12;
 	       // 综合损失值，越小代表分块方案越优
 	       const double penalty = aspectPenalty + areaPenalty;
 	 
 	       // 更新最优方案：核数更多直接替换；核数相同则选惩罚更小的
 	       if (used > bestUsed || (used == bestUsed && penalty < bestPenalty)) {
 	         bestUsed = used;
 	         bestPenalty = penalty;
 	         best = {singleM, singleN, actualMParts, actualNParts, used};
 	       }
 	     }
 	   }
 	   return best;
 	 }
 	 
 	 // 根据单核M块大小选择Cube硬件基础分块M尺寸
 	 inline int32_t SelectBaseM(uint32_t singleM) {
 	   if (singleM >= 128U) return 128;
 	   if (singleM >= 64U) return 64;
 	   if (singleM >= 32U) return 32;
 	   return 16;
 	 }
 	 
 	 // 根据单核N块大小选择Cube硬件基础分块N尺寸
 	 inline int32_t SelectBaseN(uint32_t singleN) {
 	   if (singleN >= 256U) return 256;
 	   if (singleN >= 128U) return 128;
 	   if (singleN >= 64U) return 64;
 	   if (singleN >= 32U) return 32;
 	   return 16;
 	 }
 	 
 	 } // namespace
 	 
 	 // 8字节对齐，host侧传给device的tiling参数结构体
 	 #pragma pack(push, 8)
 	 struct alignas(8) QmmCustomTilingData {
 	   TCubeTiling cubeTilingData; // CANN内置Matmul Tiling硬件参数
 	   uint32_t isPertoken;        // 标记是否开启逐Token反量化分支（1开启，0关闭）
 	   uint32_t M;                 // 矩阵输入行维度
 	   uint32_t N;                 // 输出列维度
 	   uint32_t K;                 // 隐层中间维度
 	   uint32_t singleCoreM;       // 单个逻辑核实际处理M块（Tiling API微调后真实值）
 	   uint32_t singleCoreN;       // 单个逻辑核实际处理N块（Tiling API微调后真实值）
 	   uint32_t mBlocks;           // M维度总分块数
 	   uint32_t nBlocks;           // N维度总分块数
 	   uint32_t vecTileElems;      // AIV向量单次计算元素上限
 	   uint32_t reserved;          // 预留填充位，保证对齐
 	   uint64_t userWorkspaceOffset;// 用户自定义临时buffer在总workspace内偏移
 	   uint64_t userWorkspaceSize; // 逐Token场景所需int32临时矩阵空间大小
 	   uint64_t workspaceSize;     // 总workspace大小（系统matmul缓存+用户缓存）
 	 };
 	 #pragma pack(pop)
 	 
 	 // Host侧Tiling计算入口，适配CANN 8.5.2、昇腾910C(dav-2201)
 	 static QmmCustomTilingData CalcQmmTiling(uint32_t isPertoken, uint32_t M,
 	                                          uint32_t N, uint32_t K) {
 	   QmmCustomTilingData tilingData{};
 	   // 获取昇腾平台硬件管理器实例
 	   auto ascendcPlatform =
 	       platform_ascendc::PlatformAscendCManager::GetInstance();
 	   TORCH_CHECK(ascendcPlatform != nullptr,
 	               "QmmCustom: failed to get AscendC platform information.");
 	 
 	   // 获取硬件可用AIC逻辑核总数
 	   const uint32_t aicNum = ascendcPlatform->GetCoreNumAic();
 	   TORCH_CHECK(aicNum > 0U, "QmmCustom: no available AIC core.");
 	 
 	   // 执行M/N多核分块规划
 	   const GridPlan plan = ChooseGrid(M, N, aicNum);
 	 
 	   // 初始化官方Matmul Tiling工具类
 	   matmul_tiling::MultiCoreMatmulTiling tilingApi(*ascendcPlatform);
 	   TORCH_CHECK(tilingApi.SetDim(plan.usedCores) != -1,
 	               "QmmCustom: SetDim failed.");
 	   // 设置A矩阵：全局内存GM、ND标准格式、INT8、无转置
 	   TORCH_CHECK(
 	       tilingApi.SetAType(matmul_tiling::TPosition::GM,
 	                          matmul_tiling::CubeFormat::ND,
 	                          matmul_tiling::DataType::DT_INT8, false) != -1,
 	       "QmmCustom: SetAType failed.");
 	   // 设置B权重矩阵：全局内存GM、FRACTAL_NZ压缩格式、INT8、无转置
 	   TORCH_CHECK(
 	       tilingApi.SetBType(matmul_tiling::TPosition::GM,
 	                          matmul_tiling::CubeFormat::NZ,
 	                          matmul_tiling::DataType::DT_INT8, false) != -1,
 	       "QmmCustom: SetBType failed.");
 	   // 设置输出C矩阵：GM、ND、INT32累加结果
 	   TORCH_CHECK(
 	       tilingApi.SetCType(matmul_tiling::TPosition::GM,
 	                          matmul_tiling::CubeFormat::ND,
 	                          matmul_tiling::DataType::DT_INT32) != -1,
 	       "QmmCustom: SetCType failed.");
 	   // 设置Bias参数类型（本算子关闭bias仅做声明）
 	   TORCH_CHECK(
 	       tilingApi.SetBiasType(matmul_tiling::TPosition::GM,
 	                             matmul_tiling::CubeFormat::ND,
 	                             matmul_tiling::DataType::DT_INT32) != -1,
 	       "QmmCustom: SetBiasType failed.");
 	 
 	   // 设置原始矩阵shape与分块shape
 	   TORCH_CHECK(tilingApi.SetOrgShape(M, N, K) != -1,
 	               "QmmCustom: SetOrgShape failed.");
 	   TORCH_CHECK(tilingApi.SetShape(M, N, K) != -1,
 	               "QmmCustom: SetShape failed.");
 	   // 设置单核目标计算块尺寸
 	   TORCH_CHECK(
 	       tilingApi.SetSingleShape(plan.singleM, plan.singleN, K) != -1,
 	       "QmmCustom: SetSingleShape failed.");
 	   // 关闭bias计算
 	   TORCH_CHECK(tilingApi.EnableBias(false) != -1,
 	               "QmmCustom: EnableBias failed.");
 	 
 	   // 选取硬件最优基础Cube分块尺寸
 	   const int32_t baseM = SelectBaseM(plan.singleM);
 	   const int32_t baseN = SelectBaseN(plan.singleN);
 	   TORCH_CHECK(tilingApi.SetFixSplit(baseM, baseN, -1) != -1,
 	               "QmmCustom: SetFixSplit failed.");
 	   // 缓冲区空间自动分配(-1代表交由工具自动计算)
 	   TORCH_CHECK(tilingApi.SetBufferSpace(-1, -1, -1) != -1,
 	               "QmmCustom: SetBufferSpace failed.");
 	 
 	   // 执行Tiling计算，生成硬件Cube参数
 	   const int64_t ret = tilingApi.GetTiling(tilingData.cubeTilingData);
 	   TORCH_CHECK(ret != -1, "QmmCustom: Matmul tiling failed for M=", M,
 	               ", N=", N, ", K=", K, ".");
 	 
 	   // 关键：使用Tiling API返回的真实单核块，而非之前规划值
 	   // Tiling工具会根据L0/L1缓存限制微调分块，若沿用规划值会造成多核地址错乱
 	   const uint32_t actualSingleM =
 	       static_cast<uint32_t>(tilingData.cubeTilingData.singleCoreM);
 	   const uint32_t actualSingleN =
 	       static_cast<uint32_t>(tilingData.cubeTilingData.singleCoreN);
 	   TORCH_CHECK(actualSingleM > 0U && actualSingleN > 0U,
 	               "QmmCustom: invalid single-core shape returned by tiling.");
 	   // FRACTAL_NZ格式强制N块32对齐校验
 	   TORCH_CHECK((actualSingleN % kNzNAlign) == 0U,
 	               "QmmCustom: singleCoreN must be 32-aligned for INT8 "
 	               "FRACTAL_NZ, actual=", actualSingleN);
 	 
 	   // 根据真实单核块重新计算总分块数量
 	   const uint32_t actualMBlocks = CeilDivU32(M, actualSingleM);
 	   const uint32_t actualNBlocks = CeilDivU32(N, actualSingleN);
 	   const uint32_t activeCores = actualMBlocks * actualNBlocks;
 	   TORCH_CHECK(activeCores <= tilingData.cubeTilingData.usedCoreNum,
 	               "QmmCustom: Matmul usedCoreNum is smaller than required. "
 	               "required=", activeCores, ", actual=",
 	               tilingData.cubeTilingData.usedCoreNum);
 	 
 	   // 获取Matmul库所需系统工作空间大小
 	   const uint64_t systemWorkspaceSize =
 	       static_cast<uint64_t>(ascendcPlatform->GetLibApiWorkSpaceSize());
 	   // 用户自定义buffer起始偏移，向上对齐512字节
 	   const uint64_t userWorkspaceOffset =
 	       AlignUpU64(systemWorkspaceSize, kWorkspaceAlign);
 	   // 逐Token分支需要完整M*N int32临时矩阵，非逐Token无需额外空间
 	   const uint64_t userWorkspaceSize =
 	       isPertoken
 	           ? AlignUpU64(static_cast<uint64_t>(M) * N * sizeof(int32_t),
 	                        kWorkspaceAlign)
 	           : 0U;
 	 
 	   // 填充Tiling参数结构体
 	   tilingData.isPertoken = isPertoken;
 	   tilingData.M = M;
 	   tilingData.N = N;
 	   tilingData.K = K;
 	   tilingData.singleCoreM = actualSingleM;
 	   tilingData.singleCoreN = actualSingleN;
 	   tilingData.mBlocks = actualMBlocks;
 	   tilingData.nBlocks = actualNBlocks;
 	   tilingData.vecTileElems = kVecTileElems;
 	   tilingData.userWorkspaceOffset = userWorkspaceOffset;
 	   tilingData.userWorkspaceSize = userWorkspaceSize;
 	   tilingData.workspaceSize = userWorkspaceOffset + userWorkspaceSize;
 	   return tilingData;
 	 }
 	 
 	 // A矩阵类型：GM全局内存、ND标准格式、int8、无转置
 	 using QmmAType =
 	     matmul::MatmulType<TPosition::GM, CubeFormat::ND, int8_t, false>;
 	 // B权重类型：GM全局内存、FRACTAL_NZ压缩格式、int8、无转置
 	 using QmmBType =
 	     matmul::MatmulType<TPosition::GM, CubeFormat::NZ, int8_t, false>;
 	 // Cube输出累加矩阵：GM、ND、int32
 	 using QmmCType =
 	     matmul::MatmulType<TPosition::GM, CubeFormat::ND, int32_t, false>;
 	 // Bias参数类型（本算子未启用）
 	 using QmmBiasType =
 	     matmul::MatmulType<TPosition::GM, CubeFormat::ND, int32_t, false>;
 	 // MDL开启大包搬运流水线，提升GM到L1缓存吞吐；CANN8.5.2支持INT8 Matmul
 	 using QmmMatmul =
 	     matmul::Matmul<QmmAType, QmmBType, QmmCType, QmmBiasType,
 	                    CFG_MDL>;
 	 
 	 // 单个逻辑核负责的矩阵切片信息结构体，存储分块起止、内存偏移、尾块尺寸
 	 struct QmmCoreSlice {
 	   uint32_t logicCoreId; // 当前逻辑核编号
 	   uint32_t mIndex;     // 当前核M维度分块索引
 	   uint32_t nIndex;     // 当前核N维度分块索引
 	   uint32_t mStart;     // M分块全局起始行
 	   uint32_t nStart;     // N分块全局起始列
 	   uint32_t tailM;      // 当前块有效M行数（尾块小于singleCoreM）
 	   uint32_t tailN;      // 当前块有效N列数（尾块小于singleCoreN）
 	   uint64_t aOffset;    // A矩阵GM内存偏移
 	   uint64_t bOffset;    // B权重矩阵GM内存偏移（FRACTAL_NZ专用计算）
 	   uint64_t cOffset;    // 输出C矩阵GM内存偏移
 	 };
 	 
 	 // 根据当前核编号计算本核对应的矩阵切片信息
 	 __aicore__ inline QmmCoreSlice MakeCoreSlice(
 	     const QmmCustomTilingData *__restrict tilingData) {
 	   uint32_t logicCoreId = 0;
 	   // AIC计算单元：直接取块索引作为逻辑核ID
 	   if ASCEND_IS_AIC {
 	     logicCoreId = GetBlockIdx();
 	   } else {
 	     // C/V混合模式 __mix__(1,2)：1个逻辑AIC对应2个AIV向量核，除以2映射到同一逻辑核
 	     logicCoreId = GetBlockIdx() / 2U;
 	   }
 	 
 	   QmmCoreSlice slice{};
 	   slice.logicCoreId = logicCoreId;
 	   // 二维网格排布：行优先M分块，列优先N分块
 	   slice.mIndex = logicCoreId % tilingData->mBlocks;
 	   slice.nIndex = logicCoreId / tilingData->mBlocks;
 	   // 计算当前块全局起始坐标
 	   slice.mStart = slice.mIndex * tilingData->singleCoreM;
 	   slice.nStart = slice.nIndex * tilingData->singleCoreN;
 	   // 计算有效尾块长度，防止越界
 	   slice.tailM =
 	       (slice.mStart < tilingData->M)
 	           ? min(tilingData->singleCoreM, tilingData->M - slice.mStart)
 	           : 0U;
 	   slice.tailN =
 	       (slice.nStart < tilingData->N)
 	           ? min(tilingData->singleCoreN, tilingData->N - slice.nStart)
 	           : 0U;
 	   // A矩阵 [M,K] 行优先偏移
 	   slice.aOffset = static_cast<uint64_t>(slice.mStart) * tilingData->K;
 	   // B矩阵逻辑shape [K,N] FRACTAL_NZ，N偏移对应物理偏移 nStart*K
 	   slice.bOffset = static_cast<uint64_t>(slice.nStart) * tilingData->K;
 	   // 输出C矩阵 [M,N] 行优先偏移
 	   slice.cOffset = static_cast<uint64_t>(slice.mStart) * tilingData->N +
 	                   slice.nStart;
 	   return slice;
 	 }
 	 
 	 // 分支1：基础量化矩阵乘 A8W8 -> INT32，仅AIC Cube单元执行，无后处理反量化
 	 class QmmCubeBasicKernel {
 	 public:
 	   __aicore__ inline QmmCubeBasicKernel() {}
 	 
 	   // 初始化全局GM张量、分块切片、Tiling参数
 	   __aicore__ inline void Init(
 	       GM_ADDR x1, GM_ADDR x2, GM_ADDR y,
 	       const QmmCustomTilingData *__restrict tilingData) {
 	     tilingData_ = tilingData;
 	     slice_ = MakeCoreSlice(tilingData);
 	     if ASCEND_IS_AIC {
 	       // 绑定输入A、权重B、输出INT32张量全局内存
 	       x1Gm_.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x1),
 	                             tilingData->M * tilingData->K);
 	       x2Gm_.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x2),
 	                             tilingData->K * tilingData->N);
 	       yGm_.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t *>(y),
 	                            tilingData->M * tilingData->N);
 	       cubeTiling_ = tilingData->cubeTilingData;
 	     }
 	   }
 	 
 	   // Cube矩阵乘主计算流程
 	   __aicore__ inline void ProcessCube(QmmMatmul &mm) {
 	     // 无有效数据直接返回
 	     if (slice_.tailM == 0U || slice_.tailN == 0U) {
 	       return;
 	     }
 	     // 显式设置全局原始shape，保证ND/FRACTAL_NZ地址、行跨度解释正确
 	     mm.SetOrgShape(static_cast<int>(tilingData_->M),
 	                    static_cast<int>(tilingData_->N),
 	                    static_cast<int>(tilingData_->K));
 	     // 绑定当前核分块A、B矩阵GM地址
 	     mm.SetTensorA(x1Gm_[slice_.aOffset], false);
 	     mm.SetTensorB(x2Gm_[slice_.bOffset], false);
 	     // 设置当前块有效尾块尺寸
 	     mm.SetTail(static_cast<int>(slice_.tailM),
 	                static_cast<int>(slice_.tailN),
 	                static_cast<int>(tilingData_->K));
 	     // 执行完整矩阵乘流水线，结果写入输出GM对应切片
 	     mm.IterateAll(yGm_[slice_.cOffset]);
 	     mm.End();
 	   }
 	 
 	   TCubeTiling cubeTiling_;
 	 
 	 private:
 	   const QmmCustomTilingData *tilingData_{nullptr};
 	   QmmCoreSlice slice_{};
 	   GlobalTensor<int8_t> x1Gm_;  // 输入A全局张量
 	   GlobalTensor<int8_t> x2Gm_;  // 权重B全局张量
 	   GlobalTensor<int32_t> yGm_;  // INT32累加输出全局张量
 	 };
 	 
 	 // 分支2：带逐Token反量化流水线
 	 // 流程：AIC Cube计算A8W8->INT32写入临时workspace；同组两个AIV向量核分摊M行做反量化转BF16
 	 class QmmPertokenKernel {
 	 public:
 	   __aicore__ inline QmmPertokenKernel() {}
 	 
 	   // 初始化AIC/AIV各自所需全局张量、流水线队列
 	   __aicore__ inline void Init(
 	       GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR pertokenScale, GM_ADDR y,
 	       GM_ADDR workspace, const QmmCustomTilingData *__restrict tilingData,
 	       TPipe *pipe) {
 	     tilingData_ = tilingData;
 	     slice_ = MakeCoreSlice(tilingData);
 	 
 	     if ASCEND_IS_AIC {
 	       // AIC仅需要输入A、权重B张量
 	       x1Gm_.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x1),
 	                             tilingData->M * tilingData->K);
 	       x2Gm_.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x2),
 	                             tilingData->K * tilingData->N);
 	       cubeTiling_ = tilingData->cubeTilingData;
 	     }
 	 
 	     if ASCEND_IS_AIV {
 	       // AIV向量核绑定临时int32缓存、通道scale、逐token scale、BF16输出
 	       GM_ADDR userWorkspace = workspace + tilingData->userWorkspaceOffset;
 	       workspaceGm_.SetGlobalBuffer(
 	           reinterpret_cast<__gm__ int32_t *>(userWorkspace),
 	           tilingData->M * tilingData->N);
 	       scaleGm_.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(scale),
 	                                tilingData->N);
 	       pertokenScaleGm_.SetGlobalBuffer(
 	           reinterpret_cast<__gm__ float *>(pertokenScale), tilingData->M);
 	       yGm_.SetGlobalBuffer(reinterpret_cast<__gm__ bfloat16_t *>(y),
 	                            tilingData->M * tilingData->N);
 	 
 	       // 初始化流水线队列：int32中间结果、通道scale、BF16输出、浮点计算缓存
 	       pipe->InitBuffer(mmInQueue_, 1,
 	                        tilingData->vecTileElems * sizeof(int32_t));
 	       pipe->InitBuffer(scaleQueue_, 1,
 	                        tilingData->vecTileElems * sizeof(float));
 	       pipe->InitBuffer(outQueue_, 1,
 	                        tilingData->vecTileElems * sizeof(bfloat16_t));
 	       pipe->InitBuffer(calcBuf_,
 	                        tilingData->vecTileElems * sizeof(float));
 	     }
 	   }
 	 
 	   // AIC Cube计算逻辑，结果写入workspace临时int32矩阵，完成后发同步标记唤醒AIV
 	   __aicore__ inline void ProcessCube(QmmMatmul &mm, GM_ADDR workspace) {
 	     if (slice_.tailM == 0U || slice_.tailN == 0U) {
 	       CrossCoreSetFlag<0x2, PIPE_FIX>(kCubeToVectorFlag);
 	       return;
 	     }
 	 
 	     GM_ADDR userWorkspace = workspace + tilingData_->userWorkspaceOffset;
 	     GlobalTensor<int32_t> workspaceGm;
 	     workspaceGm.SetGlobalBuffer(
 	         reinterpret_cast<__gm__ int32_t *>(userWorkspace),
 	         tilingData_->M * tilingData_->N);
 	 
 	     // 全局shape配置保证格式与偏移计算正确
 	     mm.SetOrgShape(static_cast<int>(tilingData_->M),
 	                    static_cast<int>(tilingData_->N),
 	                    static_cast<int>(tilingData_->K));
 	     mm.SetTensorA(x1Gm_[slice_.aOffset], false);
 	     mm.SetTensorB(x2Gm_[slice_.bOffset], false);
 	     mm.SetTail(static_cast<int>(slice_.tailM),
 	                static_cast<int>(slice_.tailN),
 	                static_cast<int>(tilingData_->K));
 	     // Cube结果写入全局临时workspace
 	     mm.IterateAll(workspaceGm[slice_.cOffset]);
 	     mm.End();
 	 
 	     // PIPE_FIX同步：Matmul GM写完成后再通知同组两个AIV向量核开始反量化
 	     CrossCoreSetFlag<0x2, PIPE_FIX>(kCubeToVectorFlag);
 	   }
 	 
 	   // AIV向量核反量化计算流水线：int32 -> float*通道scale*token scale -> BF16
 	   __aicore__ inline void ProcessVector() {
 	     // 阻塞等待AIC Cube计算完成标记
 	     CrossCoreWaitFlag(kCubeToVectorFlag);
 	 
 	     if (slice_.tailM == 0U || slice_.tailN == 0U) {
 	       return;
 	     }
 	 
 	     // 两个AIV子核平分当前M块行，subBlock 0取前半行，1取后半行
 	     const uint32_t subBlock = GetSubBlockIdx() & 1U;
 	     const uint32_t firstRows = (slice_.tailM + 1U) / 2U;
 	     const uint32_t rowOffset = (subBlock == 0U) ? 0U : firstRows;
 	     const uint32_t rowCount =
 	         (subBlock == 0U) ? firstRows : (slice_.tailM - firstRows);
 	 
 	     // 遍历分配到当前AIV的所有行
 	     for (uint32_t localRow = 0; localRow < rowCount; ++localRow) {
 	       const uint32_t globalRow = slice_.mStart + rowOffset + localRow;
 	       // 当前行逐Token缩放系数
 	       const float tokenScale = pertokenScaleGm_.GetValue(globalRow);
 	 
 	       // N维度分块向量化计算，每块最多vecTileElems个元素
 	       for (uint32_t col = 0; col < slice_.tailN;
 	            col += tilingData_->vecTileElems) {
 	         const uint32_t curN =
 	             min(tilingData_->vecTileElems, slice_.tailN - col);
 	         // 临时矩阵全局内存偏移
 	         const uint64_t matrixOffset =
 	             static_cast<uint64_t>(globalRow) * tilingData_->N +
 	             slice_.nStart + col;
 	         // 通道scale偏移
 	         const uint64_t scaleOffset = slice_.nStart + col;
 	 
 	         // 从GM拷贝int32累加值、通道scale到本地队列
 	         LocalTensor<int32_t> mmLocal =
 	             mmInQueue_.AllocTensor<int32_t>();
 	         LocalTensor<float> scaleLocal =
 	             scaleQueue_.AllocTensor<float>();
 	         DataCopy(mmLocal, workspaceGm_[matrixOffset], curN);
 	         DataCopy(scaleLocal, scaleGm_[scaleOffset], curN);
 	         mmInQueue_.EnQue(mmLocal);
 	         scaleQueue_.EnQue(scaleLocal);
 	 
 	         // 取出本地张量执行向量运算
 	         mmLocal = mmInQueue_.DeQue<int32_t>();
 	         scaleLocal = scaleQueue_.DeQue<float>();
 	         LocalTensor<float> calcLocal = calcBuf_.Get<float>();
 	         LocalTensor<bfloat16_t> outLocal =
 	             outQueue_.AllocTensor<bfloat16_t>();
 	 
 	         // 计算流程：int32转float → 乘通道scale → 乘逐token scale → 转BF16
 	         Cast(calcLocal, mmLocal, RoundMode::CAST_NONE, curN);
 	         Mul(calcLocal, calcLocal, scaleLocal, curN);
 	         Muls(calcLocal, calcLocal, tokenScale, curN);
 	         Cast(outLocal, calcLocal, RoundMode::CAST_RINT, curN);
 	 
 	         // 结果写回全局输出GM
 	         outQueue_.EnQue(outLocal);
 	         outLocal = outQueue_.DeQue<bfloat16_t>();
 	         DataCopy(yGm_[matrixOffset], outLocal, curN);
 	 
 	         // 释放本地张量内存
 	         outQueue_.FreeTensor(outLocal);
 	         scaleQueue_.FreeTensor(scaleLocal);
 	         mmInQueue_.FreeTensor(mmLocal);
 	       }
 	     }
 	   }
 	 
 	   TCubeTiling cubeTiling_;
 	 
 	 private:
 	   const QmmCustomTilingData *tilingData_{nullptr};
 	   QmmCoreSlice slice_{};
 	   GlobalTensor<int8_t> x1Gm_;                 // 输入A int8 GM张量
 	   GlobalTensor<int8_t> x2Gm_;                 // 权重B int8 GM张量
 	   GlobalTensor<int32_t> workspaceGm_;         // Cube临时int32缓存GM
 	   GlobalTensor<float> scaleGm_;               // 通道维度缩放系数float
 	   GlobalTensor<float> pertokenScaleGm_;       // 逐Token行缩放系数float
 	   GlobalTensor<bfloat16_t> yGm_;             // 最终BF16输出张量
 	   TQue<QuePosition::VECIN, 1> mmInQueue_;     // 向量输入队列：int32中间结果
 	   TQue<QuePosition::VECIN, 1> scaleQueue_;    // 向量输入队列：通道scale
 	   TQue<QuePosition::VECOUT, 1> outQueue_;     // 向量输出队列：BF16结果
 	   TBuf<QuePosition::VECCALC> calcBuf_;        // 向量计算本地浮点缓存
 	 };
 	 
 	 // 混合核全局Kernel入口：__mix__(1,2) 1个AIC搭配2个AIV向量核
 	 __global__ __mix__(1, 2) __aicore__
 	 void qmm_custom_kernel(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale,
 	                        GM_ADDR pertoken_scale, GM_ADDR y, GM_ADDR workspace,
 	                        QmmCustomTilingData tilingData) {
 	   // 声明混合核任务类型：1AIC+2AIV
 	   KERNEL_TASK_TYPE_DEFAULT(KERNEL_TYPE_MIX_AIC_1_2);
 	   if (workspace == nullptr) {
 	     return;
 	   }
 	 
 	   TPipe pipe;
 	   if ASCEND_IS_AIC {
 	     // 给Matmul算子绑定系统工作空间缓存
 	     SetSysWorkspace(workspace);
 	     if (GetSysWorkSpacePtr() == nullptr) {
 	       return;
 	     }
 	   }
 	 
 	   // 分支1：无逐Token反量化，仅AIC执行Cube乘，输出INT32
 	   if (tilingData.isPertoken == 0U) {
 	     QmmCubeBasicKernel kernel;
 	     kernel.Init(x1, x2, y, &tilingData);
 	     if ASCEND_IS_AIC {
 	       QmmMatmul mm;
 	       // 注册Matmul计算对象到流水线，绑定系统缓存与Tiling参数
 	       REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), mm,
 	                         &kernel.cubeTiling_);
 	       kernel.ProcessCube(mm);
 	     }
 	   } else {
 	     // 分支2：带逐Token反量化混合流水线
 	     QmmPertokenKernel kernel;
 	     kernel.Init(x1, x2, scale, pertoken_scale, y, workspace, &tilingData,
 	                 &pipe);
 	     if ASCEND_IS_AIC {
 	       // AIC执行Cube量化矩阵乘，写入临时workspace
 	       QmmMatmul mm;
 	       REGIST_MATMUL_OBJ(&pipe, GetSysWorkSpacePtr(), mm,
 	                         &kernel.cubeTiling_);
 	       kernel.ProcessCube(mm, workspace);
 	     }
 	     if ASCEND_IS_AIV {
 	       // AIV向量核执行反量化、类型转换后输出BF16
 	       kernel.ProcessVector();
 	     }
 	   }
 	   // 销毁流水线资源
 	   pipe.Destroy();
 	 }
 	 
 	 namespace ascendc_ops {
 	 
 	 // PyTorch侧Host调用入口，自定义QMM量化矩阵乘算子
 	 at::Tensor qmm_custom(const at::Tensor &x1, const at::Tensor &x2,
 	                       const at::Tensor &scale,
 	                       const c10::optional<at::Tensor> &pertoken_scale) {
 	   // 校验输入张量设备为昇腾NPU（PrivateUse1）
 	   TORCH_CHECK(x1.device().type() == c10::DeviceType::PrivateUse1,
 	               "QmmCustom: x1 must be an NPU tensor.");
 	   TORCH_CHECK(x2.device().type() == c10::DeviceType::PrivateUse1,
 	               "QmmCustom: x2 must be an NPU tensor.");
 	   TORCH_CHECK(scale.device().type() == c10::DeviceType::PrivateUse1,
 	               "QmmCustom: scale must be an NPU tensor.");
 	   // 输入必须是二维矩阵 [M,K] * [K,N]
 	   TORCH_CHECK(x1.dim() == 2 && x2.dim() == 2,
 	               "QmmCustom: x1 and x2 must be 2-D logical tensors.");
 	   // 输入A、权重B必须是int8量化类型
 	   TORCH_CHECK(x1.scalar_type() == at::ScalarType::Char &&
 	                   x2.scalar_type() == at::ScalarType::Char,
 	               "QmmCustom: x1/x2 dtype must be int8.");
 	   // 通道缩放系数float32
 	   TORCH_CHECK(scale.scalar_type() == at::ScalarType::Float,
 	               "QmmCustom: scale dtype must be float32.");
 	 
 	   // 获取矩阵维度 M K N
 	   const int64_t M64 = x1.size(0);
 	   const int64_t K64 = x1.size(1);
 	   const int64_t N64 = x2.size(1);
 	   // 矩阵乘维度匹配校验 x1[M,K] x2[K,N]
 	   TORCH_CHECK(x2.size(0) == K64,
 	               "QmmCustom: x1.size(1) must equal x2.size(0).");
 	   TORCH_CHECK(M64 > 0 && N64 > 0 && K64 > 0,
 	               "QmmCustom: M/N/K must be positive.");
 	   // 维度数值不超过uint32范围，适配Device侧参数类型
 	   TORCH_CHECK(M64 <= std::numeric_limits<uint32_t>::max() &&
 	                   N64 <= std::numeric_limits<uint32_t>::max() &&
 	                   K64 <= std::numeric_limits<uint32_t>::max(),
 	               "QmmCustom: M/N/K exceeds uint32 range.");
 	   // FRACTAL_NZ格式强制K、N维度32对齐
 	   TORCH_CHECK((K64 % kNzKAlign) == 0,
 	               "QmmCustom: K must be a multiple of 32 for INT8 FRACTAL_NZ.");
 	   TORCH_CHECK((N64 % kNzNAlign) == 0,
 	               "QmmCustom: N must be a multiple of 32 for INT8 "
 	               "FRACTAL_NZ.");
 	   // 通道scale长度等于输出列N
 	   TORCH_CHECK(scale.numel() == N64,
 	               "QmmCustom: scale.numel() must equal N.");
 	 
 	   // 判断是否开启逐Token反量化分支
 	   const uint32_t isPertoken =
 	       pertoken_scale.has_value() && pertoken_scale.value().defined() ? 1U : 0U;
 	   if (isPertoken) {
 	     const at::Tensor &tokenScale = pertoken_scale.value();
 	     // 逐Token缩放系数设备、类型、长度校验
 	     TORCH_CHECK(tokenScale.device().type() == c10::DeviceType::PrivateUse1,
 	                 "QmmCustom: pertoken_scale must be an NPU tensor.");
 	     TORCH_CHECK(tokenScale.scalar_type() == at::ScalarType::Float,
 	                 "QmmCustom: pertoken_scale dtype must be float32.");
 	     TORCH_CHECK(tokenScale.numel() == M64,
 	                 "QmmCustom: pertoken_scale.numel() must equal M.");
 	   }
 	 
 	   // 维度转uint32传入Device侧
 	   const uint32_t M = static_cast<uint32_t>(M64);
 	   const uint32_t N = static_cast<uint32_t>(N64);
 	   const uint32_t K = static_cast<uint32_t>(K64);
 	   // 获取当前NPU计算流
 	   auto aclStream = c10_npu::getCurrentNPUStream().stream(true);
 	 
 	   // 分支判定输出数据类型：带逐Token量化输出BF16，否则输出INT32
 	   const at::ScalarType outputDtype =
 	       isPertoken ? at::ScalarType::BFloat16 : at::ScalarType::Int;
 	   at::Tensor y = at::empty({M64, N64}, x1.options().dtype(outputDtype));
 	 
 	   // Host侧计算Tiling分块参数
 	   QmmCustomTilingData tilingData = CalcQmmTiling(isPertoken, M, N, K);
 	   // 获取所需逻辑核总数
 	   const uint32_t numBlocks = tilingData.cubeTilingData.usedCoreNum;
 	   TORCH_CHECK(numBlocks > 0U, "QmmCustom: invalid blockDim.");
 	 
 	   // 分配算子总工作空间buffer（Matmul系统缓存+用户临时int32矩阵）
 	   at::Tensor workspaceTensor =
 	       at::empty({static_cast<int64_t>(tilingData.workspaceSize)},
 	                 x1.options().dtype(at::ScalarType::Byte));
 	 
 	   // 逐Token缩放系数数据指针，无则传空
 	   uint8_t *pertokenPtr =
 	       isPertoken
 	           ? reinterpret_cast<uint8_t *>(
 	                 pertoken_scale.value().mutable_data_ptr())
 	           : nullptr;
 	 
 	   // 直接Launch昇腾混合核Kernel，无额外host封装，适配CANN8.5.2性能采集
 	   qmm_custom_kernel<<<numBlocks, 0, aclStream>>>(
 	       reinterpret_cast<uint8_t *>(x1.mutable_data_ptr()),
 	       reinterpret_cast<uint8_t *>(x2.mutable_data_ptr()),
 	       reinterpret_cast<uint8_t *>(scale.mutable_data_ptr()), pertokenPtr,
 	       reinterpret_cast<uint8_t *>(y.mutable_data_ptr()),
 	       workspaceTensor.data_ptr<uint8_t>(), tilingData);
 	   return y;
 	 }
 	 
 	 } // namespace ascendc_ops
 	 
 	 // 注册Torch自定义算子符号
 	 TORCH_LIBRARY(ascendc_ops, m) {
 	   m.def("qmm_custom(Tensor x1, Tensor x2, Tensor scale, Tensor? "
 	         "pertoken_scale=None) -> Tensor");
 	 }
 	 
 	 // 绑定NPU后端算子实现
 	 TORCH_LIBRARY_IMPL(ascendc_ops, PrivateUse1, m) {
 	   m.impl("qmm_custom", TORCH_FN(ascendc_ops::qmm_custom));
         }

Overwriting /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/qmm_custom.asc


---
## 4 编译自定义算子

### 任务四：成功编译自定义算子


在算子目录下执行编译命令：

In [4]:
!cd {QMM_CUSTOM_DIR} && mkdir -p build && cd build && cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 .. && make -j;

CMake Warning at /home/developer/.local/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:22 (message):
  static library kineto_LIBRARY-NOTFOUND not found.
Call Stack (most recent call first):
  /home/developer/.local/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:125 (append_torchlib_if_found)
  CMakeLists.txt:14 (find_package)


-- Configuring done
-- Generating done
-- Build files have been written to: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/build
Consolidate compiler generated dependencies of target ascendc_ops
[ 50%] Building ASC object CMakeFiles/ascendc_ops.dir/qmm_custom.asc.o
/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/qmm_custom.asc:587:8: warning: 'SetSysWorkspace' is deprecated: NOTICE: SetSysWorkSpace has been deprecated and will be removed in the next version. [-Wdeprecated-declarations]
/home/developer/As

编译成功后会在build目录下生成ascendc_ops.so

---
## 5 测试自定义算子功能性能

### 任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程

使用测试脚本验证 QmmCustom 算子在不同规格输入下的精度和性能。测试脚本覆盖多组 (M, K, N) 规格，分别验证 INT32 输出（无 perTokenScale）和 BF16 输出（有 perTokenScale）两种路径。

核心接口调用如下：

```python
output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale_npu)
```

### 5.1 测试算子功能

需要测试的 cases 的 shape 如下，对应的是 Qwen3-8B 模型中实际出现的调用，**全部通过**视为算子功能OK：
| M    | K     | N     |
| ---- | ----- | ----- |
| 1    | 4096  | 4096  |
| 1    | 4096  | 6144  |
| 1    | 4096  | 24576 |
| 1    | 12288 | 4096  |
| 50   | 4096  | 4096  |
| 50   | 4096  | 6144  |
| 50   | 4096  | 24576 |
| 50   | 12288 | 4096  |
| 4096 | 4096  | 4096  |
| 4096 | 4096  | 6144  |
| 4096 | 4096  | 24576 |
| 4096 | 12288 | 4096  |

In [5]:
import torch
import torch_npu
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

torch.npu.config.allow_internal_format = True
torch.npu.config.allow_hf32 = False
torch.npu.set_compile_mode(jit_compile=False)

# ============================================================
# 测试配置: 多组 (M, K, N) 规格
# ============================================================
TEST_SHAPES = [
    (1, 4096, 4096),
    (1, 4096, 6144),
    (1, 4096, 24576),
    (1, 12288, 4096),
    (50, 4096, 4096),
    (50, 4096, 6144),
    (50, 4096, 24576),
    (50, 12288, 4096),
    (4096, 4096, 4096),
    (4096, 4096, 6144),
    (4096, 4096, 24576),
    (4096, 12288, 4096),
]


def check_close(output, ref, dtype):
    """统一的精度比对函数，根据输出 dtype 使用不同的 tolerance"""
    if dtype == torch.int32:
        # 整数运算应精确匹配，tolerance 为 0
        rtol, atol = 0, 0
    else:
        # BF16 允许浮点舍入误差
        rtol, atol = 0.01, 0.01

    output_f = output.to(torch.float32)
    ref_f = ref.to(torch.float32)
    passed = torch.allclose(output_f, ref_f, rtol=rtol, atol=atol)

    label = 'INT32' if dtype == torch.int32 else 'BF16'
    print(f"  {label} allclose验证: {'PASS' if passed else 'FAIL'}")

    if not passed:
        print(f"  output[0,0:5] = {output[0, 0:5].tolist()}")
        print(f"  ref[0,0:5]    = {ref[0, 0:5].tolist()}")

    return passed


def test_int32(M, K, N):
    """无 pertoken_scale, 纯INT8 matmul, INT32 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # 29: FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()

    output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale=None)
    output_cpu = output_npu.cpu()

    # float64 matmul 走 BLAS 优化, 比 int32 朴素乘加快很多;
    # float64 精度足够精确表示 int32 范围内的整数结果
    ref_output = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)

    assert output_cpu.shape == ref_output.shape, f"shape不一致: {output_cpu.shape} vs {ref_output.shape}"
    assert output_cpu.dtype == torch.int32, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_output, torch.int32)


def test_bf16(M, K, N):
    """有 pertoken_scale, INT8 matmul + dequant, BF16 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)
    pertoken_scale_cpu = torch.randn([M], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # 29: FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()
    pertoken_scale_npu = pertoken_scale_cpu.npu()

    output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu,
                                                  pertoken_scale=pertoken_scale_npu)
    output_cpu = output_npu.cpu()

    int32_result = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)
    ref_bf16 = (int32_result.to(torch.float32) * scale_cpu.unsqueeze(0) * pertoken_scale_cpu.unsqueeze(1)).to(torch.bfloat16)

    assert output_cpu.shape == ref_bf16.shape, f"shape不一致: {output_cpu.shape} vs {ref_bf16.shape}"
    assert output_cpu.dtype == torch.bfloat16, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_bf16, torch.bfloat16)


# ============================================================
# 主循环: 对每组 (M, K, N) 执行 INT32 和 BF16 测试
# ============================================================
int32_results = []
bf16_results = []

for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试]")
    int32_pass = test_int32(M, K, N)
    int32_results.append((M, K, N, int32_pass))

    print("[BF16 测试]")
    bf16_pass = test_bf16(M, K, N)
    bf16_results.append((M, K, N, bf16_pass))

# ============================================================
# 测试汇总
# ============================================================
print()
print("=" * 60)
print("测试汇总")
print("=" * 60)
print(f"{'规格':>24} | {'INT32':^8} | {'BF16':^8}")
print("-" * 46)
for (M, K, N, p1), (_, _, _, p2) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {'PASS' if p1 else 'FAIL':^8} | {'PASS' if p2 else 'FAIL':^8}")

int32_total = sum(p for _, _, _, p in int32_results)
bf16_total = sum(p for _, _, _, p in bf16_results)
total = len(TEST_SHAPES)
print(f"INT32: {int32_total}/{total} 通过, BF16: {bf16_total}/{total} 通过")
print("=" * 60)


/home/developer/.local/lib/python3.11/site-packages/torch_npu/__init__.py:320: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


规格1: M=1, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格2: M=1, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格3: M=1, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格4: M=1, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格5: M=50, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格6: M=50, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格7: M=50, K=4096, N=24576
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格8: M=50, K=12288, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格9: M=4096, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格10: M=4096, K=4096, N=6144
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格11: M=4096, K=4096, N=24576
[INT3

### 5.2 测试算子性能
基于torch_npu.profiler来测试算子性能，执行下面的code cell测试单算子性能数据：

In [ ]:
import os
import csv
from pathlib import Path

# ============================================================
# Profiler 配置
# ============================================================
PROF_OUTPUT_DIR = str(SRC_DIR / "prof_output")  # 采集数据的存放路径
WARMUP_ITERS = 0      # warmup 步数, 不采集数据

def parse_kernel_details(csv_path):
    """解析 kernel_details.csv, 提取 QmmCustom 每次执行的 Duration
    """
    rows = []
    with open(csv_path, newline="", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row in reader:
            name = row.get("Name", "")
            if "QmmCustom" not in name:
                continue

            # 解析 Input Shapes: 格式如 "1,4096;4096,4096;4096;"
            raw_shapes = row.get("Input Shapes", "").strip('"')
            parts = [p.strip() for p in raw_shapes.split(";")]
            x1_shape = parts[0].split(",")  # (M, K)
            x2_shape = parts[1].split(",")  # (K, N)
            M = int(x1_shape[0])
            K = int(x1_shape[1])
            N = int(x2_shape[1])

            # 从 Output Data Types 判断 dtype
            out_dtype_raw = row.get("Output Data Types", "").strip('"')
            dtype_tag = "INT32" if "INT32" in out_dtype_raw else "BF16"

            # Duration(us)
            duration_us = float(row.get("Duration(us)", "0") or "0")

            rows.append({
                "M": M, "K": K, "N": N,
                "dtype": dtype_tag,
                "duration_us": duration_us,
            })
    return rows


def find_kernel_details_csv(prof_dir):
    prof_path = Path(prof_dir)
    if not prof_path.exists():
        return None
    # 找出所有包含 ASCEND_PROFILER_OUTPUT/kernel_details.csv 的子目录
    candidates = []
    for subdir in sorted(prof_path.iterdir(), key=lambda d: d.name, reverse=True):
        csv_path = subdir / "ASCEND_PROFILER_OUTPUT" / "kernel_details.csv"
        if csv_path.exists():
            candidates.append(csv_path)
            break  # sorted reverse=True, 第一个就是最新的
    return str(candidates[0]) if candidates else None

# ============================================================
# 主流程: 先 warmup, 再开启 profiler 采集, 最后汇总
# ============================================================
int32_results = []
bf16_results = []

# warmup: 不采集数据, 让 NPU 稳定
if WARMUP_ITERS>0:
    print("=" * 60)
    print(f"Warmup: 运行 {WARMUP_ITERS} 轮不采集数据")
    print("=" * 60)
    for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
        test_int32(M, K, N)
        test_bf16(M, K, N)

# 创建 profiler
os.makedirs(PROF_OUTPUT_DIR, exist_ok=True)
profiler = torch_npu.profiler.profile(
    activities=[
        torch_npu.profiler.ProfilerActivity.NPU,
        torch_npu.profiler.ProfilerActivity.CPU,
    ],
    experimental_config=torch_npu.profiler._ExperimentalConfig(
        profiler_level=torch_npu.profiler.ProfilerLevel.Level1,
        aic_metrics=torch_npu.profiler.AiCMetrics.PipeUtilization,
    ),
    on_trace_ready=torch_npu.profiler.tensorboard_trace_handler(PROF_OUTPUT_DIR),
)

# 开启 profiler 采集
profiler.start()

for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试]")
    int32_pass = test_int32(M, K, N)
    int32_results.append((M, K, N, int32_pass))
    profiler.step()

    print("[BF16 测试]")
    bf16_pass = test_bf16(M, K, N)
    bf16_results.append((M, K, N, bf16_pass))
    profiler.step()

# 停止 profiler
profiler.stop()

# ============================================================
# 测试汇总
# ============================================================
print()
print("=" * 60)
print("测试汇总")
print("=" * 60)
print(f"{'规格':>24} | {'INT32':^8} | {'BF16':^8}")
print("-" * 46)
for (M, K, N, p1), (_, _, _, p2) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {'PASS' if p1 else 'FAIL':^8} | {'PASS' if p2 else 'FAIL':^8}")

int32_total = sum(p for _, _, _, p in int32_results)
bf16_total = sum(p for _, _, _, p in bf16_results)
total = len(TEST_SHAPES)
print(f"INT32: {int32_total}/{total} 通过, BF16: {bf16_total}/{total} 通过")

# ============================================================
# Profiler 数据汇总: 每种规格每种 dtype 的 Duration(us)
# ============================================================
print()
print("=" * 70)
print("Profiler 数据汇总 (QmmCustom Duration)")
print("=" * 70)

csv_path = find_kernel_details_csv(PROF_OUTPUT_DIR)
print(f"原始数据路径：{csv_path}")
if csv_path:
    kernel_rows = parse_kernel_details(csv_path)
    if kernel_rows:
        # 按 (M, K, N) 分组, 每组内有 INT32 和 BF16
        shape_map = {}
        for kr in kernel_rows:
            key = (kr["M"], kr["K"], kr["N"])
            shape_map.setdefault(key, {})[kr["dtype"]] = kr["duration_us"]

        print(f"{'规格 (M,K,N)':>24} | {'INT32 Duration(us)':>18} | {'BF16 Duration(us)':>18}")
        print("-" * 70)
        for M, K, N in TEST_SHAPES:
            key = (M, K, N)
            int32_dur = shape_map.get(key, {}).get("INT32", "-")
            bf16_dur = shape_map.get(key, {}).get("BF16", "-")
            int32_str = f"{int32_dur:.3f}" if isinstance(int32_dur, float) else int32_dur
            bf16_str = f"{bf16_dur:.3f}" if isinstance(bf16_dur, float) else bf16_dur
            tag = f"M={M},K={K},N={N}"
            print(f"{tag:>24} | {int32_str:>18} | {bf16_str:>18}")
    else:
        print(f"在 {csv_path} 中未找到 QmmCustom 算子数据")
else:
    print(f"未在 {PROF_OUTPUT_DIR} 中找到 kernel_details.csv 文件")
    print("请检查 profiler 输出目录")

print("=" * 70)


---
## 6 自定义算子接入模型

### 6.1 定位替换点

Qwen3-8B 推理框架中的 W8A8 量化 matmul 通过 `CompressedTensorsW8A8Int8LinearMethod` 类实现，其 `apply_weights` 方法中使用 `torch_npu.npu_quant_matmul` 执行 INT8 量化矩阵乘法。我们需要将这个调用替换为自定义的 `torch.ops.ascendc_ops.qmm_custom` 算子。

查看原始实现：

In [6]:
!grep -n 'npu_quant_matmul' {RECIPE_ROOT}/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py

94:        # --- 替换核心: npu_quant_matmul → qmm_custom ---


### 6.2 替换实现

在 `CompressedTensorsW8A8Int8LinearMethod` 中，将 `torch_npu.npu_quant_matmul` 替换为 `torch.ops.ascendc_ops.qmm_custom`。

替换后的方法为：

```python
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

def apply_weights(self, layer: torch.nn.Module,
                x: torch.Tensor,
                bias: Optional[torch.Tensor],
                dynamic_scale: Optional = None,
                out_dtype: Optional = torch.bfloat16
                ) -> Union[torch.Tensor, Dict[str, Any]]:
    if dynamic_scale is not None:
        x_scale = dynamic_scale
    else:
        x, x_scale = torch_npu.npu_dynamic_quant(x, smooth_scales=layer.smooth_scales)
    out_shape_dim_0 = x.size()[:-1]
    x = x.view(-1, x.size(-1))
    x_scale = x_scale.view(-1)

    # --- 替换核心: npu_quant_matmul → qmm_custom ---
    x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight, 
                                layer.weight_scale.view(-1),
                                pertoken_scale=None if out_dtype == torch.int32 else x_scale)
    # --- 替换结束 ---

    x = x.view(out_shape_dim_0 + (-1, ))
    if out_dtype == torch.int32:
        return x, x_scale
    else:
        return x
```

执行如下操作替换推理网络中调用的接口实现：

In [7]:
# 6.2 替换实现: 将 torch_npu.npu_quant_matmul 替换为 torch.ops.ascendc_ops.qmm_custom

# 1: 导入libascendc_ops.so
W8A8_FILE = RECIPE_ROOT / 'module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py'
print('[替换] 目标文件:', W8A8_FILE)

ct_code = W8A8_FILE.read_text(encoding='utf-8')
_lib_path = str(QMM_CUSTOM_DIR / "build/libascendc_ops.so")
_load_lib_stmt = 'torch.ops.load_library(r"' + _lib_path + '")'
if _load_lib_stmt not in ct_code:
    # 在 'import torch' 行后插入 'torch.ops.load_library'
    ct_code = ct_code.replace('import torch\n', 'import torch\n' + _load_lib_stmt + '\n')
    W8A8_FILE.write_text(ct_code, encoding='utf-8')
    print('[插入] compressed_tensors.py: 已添加 torch.ops.load_library')
else:
    print('[插入] compressed_tensors.py: torch.ops.load_library 已存在，跳过')

# 2: 在 compressed_tensors_w8a8_int8.py 中替换 npu_quant_matmul → qmm_custom
code = W8A8_FILE.read_text(encoding='utf-8')

# 替换 npu_quant_matmul 调用为 qmm_custom 调用
old_call = (
    'x = torch_npu.npu_quant_matmul(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale,\n'
    '                                    bias=layer.bias,\n'
    '                                    output_dtype=out_dtype)'
)
new_call = (
    '# --- 替换核心: npu_quant_matmul → qmm_custom ---\n'
    '        x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale)\n'
    '        # --- 替换结束 ---'
)

if old_call in code:
    code = code.replace(old_call, new_call)
    W8A8_FILE.write_text(code, encoding='utf-8')
    print('[替换] 成功: npu_quant_matmul → qmm_custom')
else:
    print('[替换] 文件中未找到 npu_quant_matmul 调用，可能已替换')

# 验证替换结果
result = W8A8_FILE.read_text(encoding='utf-8')
if 'torch.ops.ascendc_ops.qmm_custom' in result:
    print('[验证] 替换已生效，当前使用 qmm_custom 算子')
else:
    print('[验证] 替换未生效，请检查文件内容')


[替换] 目标文件: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py
[插入] compressed_tensors.py: torch.ops.load_library 已存在，跳过
[替换] 文件中未找到 npu_quant_matmul 调用，可能已替换
[验证] 替换已生效，当前使用 qmm_custom 算子


### 6.3 替换实现后执行量化推理

指定本地的量化模型权重路径，调用infer.sh执行量化推理，查看 W8A8 量化模型推理生成的文本，对比替换前的生成文本是否一致。

**注意：需要先完成第五章的量化模型推理过程，并将`W8A8_WEIGHT`指定为本地的量化模型权重的路径**

In [8]:
import sys, subprocess, os
from inference_scripts.recipe_workflow import prepare_runtime_yaml

# 将W8A8_WEIGHT设置为量化模型权重实际所在路径
W8A8_WEIGHT = TUTORIAL_DIR / 'src/data/models/Qwen3-8B-W8A8'

YAML_A8W8_TEMPLATE =  RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_1tp.yaml'
YAML_A8W8_CUSTOM  = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom.yaml'
prepare_runtime_yaml(YAML_A8W8_TEMPLATE, YAML_A8W8_CUSTOM, model_path=str(W8A8_WEIGHT))

env = os.environ.copy()
python_dir = os.path.dirname(sys.executable)
env["PATH"] = python_dir + ":" + env.get("PATH", "")
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_CUSTOM],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

rm: cannot remove '/root/atc_data/': Permission denied


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom.yaml)
world_size = 1
[INFO][2026-07-23 16:29:06.053][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-07-23 16:29:06.053][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-07-23 16:29:06.054][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-07-23 16:29:06.056][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-07-23 16:29:06.056][patch_getenv.py:15] get env RES_PATH = res/20260723/qwen3_8b_qwen3_8b_a8w8_custom
[INFO][2026-07-23 16:29:06.056][infer.py:146] Inference Configuration
[INFO][2026-07-23 16:29:06.056][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/gitCode/cann/cann-lear

---
## 7 模型性能测试

### 任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程

将 QmmCustom 算子接入模型后，使用 Profiling 工具观测模型推理性能，对比自定义算子替换前后的 matmul 耗时变化。

### 7.1 准备 Profiling YAML

沿用 recipes 的 YAML + `infer.sh` 工作流，在配置中打开 Profiler，运行一次推理并收集性能数据。

通过如下操作将 YAML 配置中的enable_profiler配置为 True ：

In [9]:
from inference_scripts.recipe_workflow import prepare_profiling_yaml

YAML_A8W8_CUSTOM_PROFILER = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml'

prepare_profiling_yaml(YAML_A8W8_CUSTOM, YAML_A8W8_CUSTOM_PROFILER, enable_profiler=True)

{'model_config': {'model_name': 'qwen3_8b',
  'model_path': '/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8',
  'exe_mode': 'eager',
  'enable_profiler': True,
  'with_ckpt': True,
  'enable_cache_compile': False,
  'enable_static_kernel': False},
 'data_config': {'dataset': 'default', 'input_truncated_len': 4096},
 'parallel_config': {'world_size': 1,
  'attn_tp_size': 1,
  'moe_tp_size': 1,
  'embed_tp_size': 1,
  'lmhead_tp_size': 1},
 'scheduler_config': {'max_new_tokens': 256, 'batch_size': 1}}

### 7.2 启动 Profiling 推理

使用 recipes 命令启动带 Profiler 的推理，推理结束后会在结果目录中生成 `prof/` 产物。

In [10]:
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_CUSTOM_PROFILER],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

rm: cannot remove '/root/atc_data/': Permission denied


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml)
world_size = 1
[INFO][2026-07-23 16:30:32.724][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-07-23 16:30:32.724][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-07-23 16:30:32.724][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-07-23 16:30:32.727][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-07-23 16:30:32.727][patch_getenv.py:15] get env RES_PATH = res/20260723/qwen3_8b_qwen3_8b_a8w8_custom_profiler
[INFO][2026-07-23 16:30:32.727][infer.py:146] Inference Configuration
[INFO][2026-07-23 16:30:32.727][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/gitC

### 7.3 查看 Profiling 结果

推理完成后，定位 `prof/` 目录，重点关注其中的 `op_statistic.csv` 和 `kernel_details.csv` 这两个文件。`op_statistic.csv`中展示了模型调用的所有算子的统计信息，可快速确认自定义算子 QmmCustom 的调用情况。`kernel_details.csv`中展示了详细的接口调用信息，可自行查看筛选自己关注的算子的性能。

通过如下操作统计 `kernel_details.csv` 中的 QmmCustom 算子耗时，可自行对比自定义算子替换前后的性能差异。

In [11]:
import pandas as pd
from pathlib import Path

RES_DIR = RECIPE_ROOT / 'models/qwen/res'

# 取最新的 profiling 结果目录
result_dirs = sorted([d for d in RES_DIR.iterdir() if d.is_dir()], key=lambda d: d.name, reverse=True)
latest_result = result_dirs[0]
print(f'最新结果目录: {latest_result}')

def find_kernel_csv(phase_dir):
    matches = sorted(phase_dir.rglob('kernel_details.csv'), key=lambda p: p.parent.parent.name, reverse=True)
    return matches[0] if matches else None

prefill_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill')
decode_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/decode')
assert prefill_csv and decode_csv, 'kernel_details.csv 未找到，请确认 profiling 已完成'
print(f'Prefill CSV: {prefill_csv}')
print(f'Decode CSV: {decode_csv}')

# 读取 CSV 并筛选 QmmCustom 算子
prefill_df = pd.read_csv(prefill_csv)
decode_df = pd.read_csv(decode_csv)

prefill_qmm = prefill_df[prefill_df['Type'].str.contains('QmmCustom')].copy()
decode_qmm = decode_df[decode_df['Type'].str.contains('QmmCustom')].copy()

# 按 Input Shapes 分组，统计每种 shape 下的平均 Duration
def qmm_duration_stats(df, phase):
    stats = df.groupby(['Input Shapes', 'Output Data Types'])['Duration(us)'].agg(['mean', 'count']).reset_index()
    stats.columns = ['Input Shapes', 'Output Dtype', 'Avg Duration(us)', 'Calls']
    stats = stats.sort_values('Avg Duration(us)', ascending=False).reset_index(drop=True)
    stats.insert(0, 'Phase', phase)
    return stats

prefill_stats = qmm_duration_stats(prefill_qmm, 'Prefill')
decode_stats = qmm_duration_stats(decode_qmm, 'Decode')

combined = pd.concat([prefill_stats, decode_stats], ignore_index=True)
display(combined.style.set_caption('QmmCustom 各 Shape 平均耗时 (us)'))

最新结果目录: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723
Prefill CSV: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill/f14d07c7168f46e6a4f9f239e38f78a3_9916_20260723150637867_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv
Decode CSV: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260723/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/decode/f14d07c7168f46e6a4f9f239e38f78a3_9916_20260723150641414_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv


,Phase,Input Shapes,Output Dtype,Avg Duration(us),Calls


## 8 提交要求

> 完整的提交说明参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

1. 提交完整的 Jupyter Notebook（.ipynb），**保留所有单元格输出**。
2. 提交前依次运行所有单元格，确保：
   - 自定义算子编译成功；
   - 算子功能测试通过；
   - 性能数据收集成功。
3. **文件命名**：`gitcode账号_result.ipynb`。
4. **允许并且鼓励**使用 AI 工具辅助，但须独立思考并在报告中简述（解决了什么疑惑、是否引入新 bug）。
5. **不得抄袭。**

---
### 小结

本章完成了 QmmCustom 算子的开发、编译和单算子功能性能测试，并将该自定义算子接入了 Qwen3-8B 量化模型，验证功能并观测了其在模型中的性能。


### 附录
实践相关参考材料链接：
- [Ascend C 高阶api文档](https://gitcode.com/cann/asc-devkit/blob/master/docs/api/SIMD-API/%E9%AB%98%E9%98%B6API/%E9%AB%98%E9%98%B6API.md)
- [昇腾社区文档](https://www.hiascend.com/zh/document)
- [Ascend C 算子开发系列教程](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development)
- [Ascend C 算子功能调试](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development/07_Troubleshooting)
- [Matmul最佳实践样例](https://gitcode.com/cann/asc-devkit/blob/master/examples/01_simd_cpp_api/05_best_practices/01_matrix_compute/matmul_high_performance/README.md)
- [CANN内置量化矩阵乘算子实现](https://gitcode.com/cann/ops-nn/tree/master/matmul/quant_batch_matmul_v3)